# Tutorial: Latent Space Explorer App

**Purpose:** Guide to using the standalone Streamlit application for interactive latent space exploration
**Author:** Hallett Lab
**Date:** 2026-01-29

This tutorial covers the interactive Latent Space Explorer app, which allows you to:
- Visualize latent spaces of trained VAE models
- Click on data points to view corresponding images
- Decode from empty space positions using k-NN interpolation
- Generate interpolations between two points

---

## 1. Launching the App

The Latent Space Explorer is a standalone Streamlit application. Launch it from the terminal:

```bash
cd <repo>
streamlit run src/candescence/interface/apps/latent_explorer_app.py
```

This will open a browser window (typically at http://localhost:8501).

## 2. Data Loading Options

The app supports four data loading methods:

### 2.1 From Model + Images (Recommended)

This is the **recommended** method for loading data from candescence_new:

1. Select a trained model from the dropdown:
   - `tutorial_tendril_vae/test_run_v1` - Tutorial training run
   - `0_37_tendrils_reproduction/manuscript_v1` - 37 tendrils reproduction

2. Images are loaded from `data/sample/images/`

3. Adjust "Max samples to load" (500-1000 recommended for responsiveness)

4. Click "Load Model + Compute Embeddings"

The app will:
- Load the trained model weights
- Load raw images and resize to 128x128
- Compute latent embeddings using the encoder
- Optionally load conditioning values from metadata

### 2.2 From Inference Object (Legacy)

Load pre-computed embeddings from candescence_master pickle files.

**Note:** This may have dependency issues (`skbio`, `inference` modules). Use "From Model + Images" instead.

### 2.3 From NPZ + Images

Load pre-extracted embeddings from `.npz` files:
- Embeddings: NumPy archive with shape (N, latent_dim)
- Images: NumPy array of shape (N, C, H, W)
- Metadata: Optional CSV file

### 2.4 Demo Data

Generate synthetic data for testing the interface without loading real data.

## 3. Visualization Controls

Once data is loaded, configure the visualization in the sidebar:

### Projection Method
- **PCA**: Fast, linear projection (recommended for initial exploration)
- **t-SNE**: Non-linear, preserves local structure (slower)
- **UMAP**: Non-linear, preserves global structure (requires umap-learn)

### Color By
Color points by any numeric metadata column (e.g., `average_hue`, `cluster`).

## 4. Exploring the Latent Space

### 4.1 Click on a Data Point

1. Click on any point in the scatter plot
2. The corresponding image appears in the "Clicked Point" panel on the right
3. The point is highlighted with an orange ring

Information displayed:
- Index number
- 2D coordinates in the projection
- The actual image
- Metadata (expandable)

### 4.2 Manual Selection

If click events don't work (older Streamlit versions), use manual selection:

1. Hover over a point to see its index in the tooltip
2. Enter the index in "Point index" field
3. The preview image updates automatically

### 4.3 Add to Selection

Click "Add to Selection" to save a point for interpolation. You can select up to 2 points.

## 5. Interpolation

Generate smooth transitions between two points in latent space.

### 5.1 Using Selected Points

1. Click on a point and "Add to Selection" (Point A)
2. Click on another point and "Add to Selection" (Point B)
3. The coordinates are shown in the Interpolation section
4. Adjust "Interpolation steps" (7 is a good default)
5. Click "Generate Interpolation"

### 5.2 Using Coordinates

You can also enter coordinates directly:
- **Start Point**: X start, Y start
- **End Point**: X end, Y end

This allows interpolation between arbitrary positions, not just existing data points.

### 5.3 Output

The interpolation shows:
- Endpoint images (real or decoded)
- Filmstrip of intermediate steps with alpha values (0.0 to 1.0)

## 6. Understanding k-NN Decoding

### Why k-NN?

When you click on empty space (not on a data point), the app needs to:
1. Estimate what latent vector corresponds to that 2D position
2. Decode that latent vector to an image

### The Challenge

The Tendril VAE uses **skip connections** (U-Net style architecture). This means:
- Encoding produces both a latent vector AND intermediate feature maps
- Decoding requires BOTH the latent vector AND those skip connections

### The Solution

The app uses k-NN (k=5 nearest neighbors) weighted interpolation:

1. **Find neighbors**: Locate the 5 closest data points in 2D space
2. **Weight by distance**: Closer points have more influence (inverse distance weighting)
3. **Interpolate latent**: Compute weighted average of their latent vectors
4. **Get skip connections**: Encode the nearest neighbor image to set skip connections
5. **Decode**: Generate the image using the interpolated latent + skip connections

### Limitation

The decoded image is influenced by the reference image's skip connections, so it's an approximation rather than a "pure" sample from that latent position.

## 7. Tips and Troubleshooting

### Performance
- Start with **500-1000 samples** for responsive exploration
- Use **PCA** for initial exploration (fastest)
- Switch to **UMAP** or **t-SNE** for final visualization

### Memory Issues
- Reduce "Max samples to load"
- Close other applications
- Use PCA instead of t-SNE/UMAP

### Click Events Not Working
- Requires Streamlit >= 1.29
- Use manual selection as fallback
- Check "Selection Event Debug" expander for diagnostics

### Module Not Found Errors
- Use "From Model + Images" instead of legacy pickle files
- Ensure candescence_new environment is activated

## 8. Programmatic Access

The app can also be launched programmatically:

In [ ]:
# Launch from Python (this will open a browser)
from candescence.interface.apps import latent_explorer_main

# Note: This runs the Streamlit server
# latent_explorer_main()

## 9. Available Models

Current trained models in candescence_new:

In [ ]:
from pathlib import Path

# List available models
model_dirs = [
    "<refined>/tutorial_tendril_vae/test_run_v1/models",
    "<refined>/0_37_tendrils_reproduction/manuscript_v1/models",
]

for model_dir in model_dirs:
    path = Path(model_dir)
    if path.exists():
        model_file = path / "model.pth"
        args_file = path / "args.json"
        print(f"Model: {path.parent.name}")
        print(f"  - model.pth: {'exists' if model_file.exists() else 'MISSING'}")
        print(f"  - args.json: {'exists' if args_file.exists() else 'MISSING'}")
        print()

## 10. Summary

The Latent Space Explorer provides an intuitive way to:

1. **Visualize** - See the structure of learned latent representations
2. **Explore** - Click to view images at any point
3. **Interpolate** - Generate smooth transitions between morphologies
4. **Understand** - Relate latent positions to visual features

**Quick Start:**
```bash
streamlit run src/candescence/interface/apps/latent_explorer_app.py
```

Then:
1. Select "From Model + Images"
2. Choose a model
3. Click "Load Model + Compute Embeddings"
4. Start clicking on points!